# momentum-buffer-update — worked example 1: Run 5 steps of SGD momentum and record the buffer trajectory

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `momentum-buffer-update`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

In classical SGD with momentum, a running buffer `b` accumulates a decayed sum of all past gradients: `b ← μ·b + g`. The parameter update then uses `b` instead of the raw gradient, so it carries memory of gradient history. The key implementation detail is that the buffer must be mutated in-place with `.copy_()` or `.mul_().add_()` so the state tensor stored in the optimizer's list is actually updated across steps.

## Worked solution

**Step 1 — initialize buffer to zero.** At the start of training the buffer is all zeros. We allocate it with `torch.zeros_like(param)`.

**Step 2 — apply the recurrence in-place each step.** `b.copy_(mu * b + g)` computes the new buffer value and writes it into the existing tensor object. This is essential: if we wrote `b = mu * b + g`, the local variable `b` would be rebound to a new tensor, but the entry in the buffer list would still point to the old zero tensor.

**Step 3 — record the L2 norm of the buffer each step.** We call `.norm().item()` after the update to get a scalar we can plot or print. The norm grows initially as gradients accumulate, then stabilises or grows more slowly.

**Step 4 — observe the trajectory.** With a constant gradient, the buffer converges geometrically to `g / (1 - μ)`. With μ=0.9 and ‖g‖=1 the buffer norm converges to 10.

In [ ]:
import torch

torch.manual_seed(0)

def momentum_trajectory(param_init, grad_fn, mu, n_steps, lr=0.01):
    """
    Run n_steps of SGD-with-momentum on a single parameter.
    grad_fn(param) -> gradient tensor at current param.
    Returns (param history, buffer norm history).
    """
    param = param_init.clone().float()
    buf = torch.zeros_like(param)
    param_history = [param.clone()]
    buf_norms = []

    for step in range(n_steps):
        g = grad_fn(param)
        # In-place buffer update: b <- mu*b + g
        buf.copy_(mu * buf + g)
        buf_norms.append(buf.norm().item())
        # Parameter update using the buffer as effective gradient
        param.data.add_(buf, alpha=-lr)
        param_history.append(param.clone())

    return param_history, buf_norms

# Constant gradient example: grad = 1 everywhere.
torch.manual_seed(3)
param0 = torch.tensor([0.0])
grad_fn = lambda p: torch.ones_like(p)

history, norms = momentum_trajectory(param0, grad_fn, mu=0.9, n_steps=5)
print("Buffer norms per step:", [f"{n:.4f}" for n in norms])
print("Param per step:       ", [f"{p.item():.4f}" for p in history])
# Buffer norm converges toward 1/(1-0.9)=10; param decreases each step.